<a href="https://colab.research.google.com/github/nivethithanm/torchcode-solutions/blob/main/P0_05a_masking_step_by_step.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FIX-01: Masking & Indexing — Step by Step

**Why you need this**: Masking shows up in EVERY attention implementation, every loss function, and every sampling method. P0-05 threw 4 concepts at you at once. This notebook takes ONE concept at a time.

**Prerequisites**: You know tensors, shapes, and basic operations (P0-01 through P0-04). That's all.

In [2]:
import torch

## Part 1: Comparison operators make boolean tensors

This is the foundation. Every mask starts with a comparison.

In [3]:
x = torch.tensor([1, 5, 3, 8, 2])

# Each comparison creates a tensor of True/False
print('x > 3  :', x > 3)    # which elements are greater than 3?
print('x == 5 :', x == 5)   # which elements equal 5?
print('x < 4  :', x < 4)    # which elements are less than 4?
print()
print('dtype:', (x > 3).dtype)  # torch.bool

x > 3  : tensor([False,  True, False,  True, False])
x == 5 : tensor([False,  True, False, False, False])
x < 4  : tensor([ True, False,  True, False,  True])

dtype: torch.bool


**Your turn**: predict the output before running each cell.

In [4]:
x = torch.tensor([[1, 2], [3, 4], [5, 6]])

# What shape is this mask? What values does it have?
mask = x > 3
print('Shape:', mask.shape)  # same as x
print('Mask:\n', mask)
# Each element independently compared to 3

Shape: torch.Size([3, 2])
Mask:
 tensor([[False, False],
        [False,  True],
        [ True,  True]])


## Part 2: Using masks to select or modify elements

Three ways to use a mask:
1. `x[mask]` — extract elements where mask is True (returns 1D)
2. `torch.where(mask, x, other)` — pick from x where True, other where False
3. `x.masked_fill(mask, value)` — fill value where mask is True

In [5]:
x = torch.tensor([10, 20, 30, 40, 50])
mask = torch.tensor([True, False, True, False, True])

# Method 1: Extract — FLAT output
selected = x[mask]
print('x[mask]:', selected)  # [10, 30, 50] — only True positions
print('Shape:', selected.shape)  # (3,) — NOT (5,)! Elements are collected into 1D

x[mask]: tensor([10, 30, 50])
Shape: torch.Size([3])


In [6]:
# Method 2: torch.where — keeps the shape
x = torch.tensor([10, 20, 30, 40, 50])
mask = x > 25

result = torch.where(mask, x, torch.tensor(0))  # keep x where True, put 0 where False
print('where(x > 25, x, 0):', result)  # [0, 0, 30, 40, 50]

# This is EXACTLY how a threshold ReLU works:
# ReLU(x) = torch.where(x > 0, x, 0)

where(x > 25, x, 0): tensor([ 0,  0, 30, 40, 50])


In [7]:
# Method 3: masked_fill — fill specific value where mask is True
x = torch.tensor([10., 20., 30., 40., 50.])
mask = x > 25

# IMPORTANT: masked_fill fills where mask is TRUE
# This is the OPPOSITE intuition from 'keep where True'
result = x.masked_fill(mask, -1.0)  # put -1 where x > 25
print('masked_fill(x>25, -1):', result)  # [10, 20, -1, -1, -1]

masked_fill(x>25, -1): tensor([10., 20., -1., -1., -1.])


**Exercise 2a**: Zero out values <= 4 in this matrix

In [8]:
x = torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=torch.float)

# Step 1: Create the mask — which elements are > 4?
keep_mask = x > 4
print('Keep mask:\n', keep_mask)

# Step 2: Use torch.where to keep those, put 0 elsewhere
# TODO: Fill in
result = torch.where(keep_mask, x, torch.tensor(0.0))

print('Result:\n', result)
expected = torch.tensor([[0, 0, 0], [0, 5, 6], [7, 8, 9]], dtype=torch.float)
assert torch.equal(result, expected)
print('✅ Passed!')

Keep mask:
 tensor([[False, False, False],
        [False,  True,  True],
        [ True,  True,  True]])
Result:
 tensor([[0., 0., 0.],
        [0., 5., 6.],
        [7., 8., 9.]])
✅ Passed!


## Part 3: Building a causal mask

In transformers, token i can only look at tokens 0, 1, ..., i (not the future).
We need a lower-triangular matrix of True/False.

Let's build it step by step.

In [9]:
# Step 1: torch.ones creates a matrix of 1s
N = 4
ones = torch.ones(N, N)
print('All ones:\n', ones)

All ones:
 tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])


In [10]:
# Step 2: torch.tril keeps only the LOWER triangle (including diagonal)
lower = torch.tril(ones)
print('Lower triangular:\n', lower)
# Row i has 1s in columns 0..i, 0s in columns i+1..N-1

Lower triangular:
 tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


In [11]:
# Step 3: Convert to boolean
causal_mask = torch.tril(torch.ones(N, N)).bool()
print('Causal mask:\n', causal_mask)
# True  = CAN attend here
# False = CANNOT attend here (future position)

Causal mask:
 tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True]])


In [12]:
# Step 4: Apply to attention scores
scores = torch.randn(N, N)
print('Raw scores:\n', scores)

# We want: where mask is False (future), fill with -inf
# masked_fill fills where condition is TRUE
# So we negate: ~causal_mask means 'is future position'
masked = scores.masked_fill(~causal_mask, float('-inf'))
print('\nMasked scores:\n', masked)
# After softmax, -inf becomes 0 probability — token can't attend to future

Raw scores:
 tensor([[ 0.7320, -2.0021, -1.3162, -0.5137],
        [ 0.1832, -1.3796,  0.3129,  0.0225],
        [-0.2627, -0.9277,  1.0343,  0.5917],
        [ 0.7993,  0.1641,  0.4489, -1.6401]])

Masked scores:
 tensor([[ 0.7320,    -inf,    -inf,    -inf],
        [ 0.1832, -1.3796,    -inf,    -inf],
        [-0.2627, -0.9277,  1.0343,    -inf],
        [ 0.7993,  0.1641,  0.4489, -1.6401]])


**Exercise 3a**: Build a causal mask for N=5 and apply it

In [15]:
N = 5
# TODO: Create causal mask
causal_mask = torch.tril(torch.ones(N, N)).bool()  # YOUR CODE (one line)

# TODO: Apply to random scores
scores = torch.randn(N, N)
masked_scores = torch.masked_fill(scores, ~causal_mask, -torch.inf)  # YOUR CODE (one line)

# Verify: future positions should be -inf
assert masked_scores[0, 1] == float('-inf')  # token 0 can't see token 1
assert masked_scores[0, 0] != float('-inf')  # token 0 CAN see itself
assert masked_scores[4, 0] != float('-inf')  # token 4 can see token 0
print('✅ Causal mask works!')

✅ Causal mask works!


## Part 4: gather — picking one element per row

**Problem**: You have logits of shape (B, num_classes) and labels of shape (B,).
You need to pick, for each batch item, the logit at the correct class index.

This is like: `for i in range(B): result[i] = logits[i, labels[i]]`

But we want to do it without a loop.

In [18]:
# Let's think about this visually
logits = torch.tensor([
    [0.1, 0.5, 0.3, 0.9],  # batch item 0: 4 classes
    [0.8, 0.2, 0.7, 0.1],  # batch item 1: 4 classes
])
labels = torch.tensor([3, 0])  # batch 0 → class 3, batch 1 → class 0

# What we want:
# batch 0: logits[0, 3] = 0.9
# batch 1: logits[1, 0] = 0.8

# With a loop (slow):
result_loop = torch.tensor([logits[i, labels[i]] for i in range(2)])
print('Loop result:', result_loop)  # [0.9, 0.8]

Loop result: tensor([0.9000, 0.8000])


In [19]:
# gather does this WITHOUT a loop.
# It needs the index to have the SAME number of dimensions as the input.

# labels is (2,) but logits is (2, 4)
# So we reshape labels to (2, 1) — we want 1 element per row
labels_2d = labels.unsqueeze(1)  # (2,) → (2, 1)
print('labels_2d:', labels_2d)     # [[3], [0]]

# gather(input, dim, index)
# dim=1 means 'gather along the columns'
# For each row i, it picks the column given by index[i, 0]
result = torch.gather(logits, dim=1, index=labels_2d)
print('gather result:', result)  # [[0.9], [0.8]]
print('shape:', result.shape)    # (2, 1)

labels_2d: tensor([[3],
        [0]])
gather result: tensor([[0.9000],
        [0.8000]])
shape: torch.Size([2, 1])


**Exercise 4a**: Use gather on a bigger example

In [22]:
B, C = 4, 10
logits = torch.randn(B, C)
labels = torch.tensor([3, 7, 1, 5])

# TODO: Step 1 — reshape labels from (B,) to (B, 1)
labels_2d = labels.unsqueeze(1)  # YOUR CODE

# TODO: Step 2 — gather the correct logit for each batch item
correct_logits = torch.gather(logits, dim=1, index=labels_2d)  # torch.gather(logits, dim=1, index=labels_2d)

# Verify
for i in range(B):
    assert correct_logits[i, 0] == logits[i, labels[i]], f'Failed for batch {i}'
print('✅ gather works!')

✅ gather works!


## Part 5: scatter_ — the reverse of gather

Gather PICKS values at indices. Scatter PLACES values at indices.

Use case: build a one-hot vector, or place top-k values back into a full-size tensor.

In [23]:
# One-hot encoding with scatter
labels = torch.tensor([2, 0, 3, 1])  # 4 samples, classes 0-3
num_classes = 4

# Start with zeros
one_hot = torch.zeros(4, num_classes)

# scatter_ puts 1.0 at the positions given by labels
one_hot.scatter_(dim=1, index=labels.unsqueeze(1), value=1.0)
# For each row i, put 1.0 at column labels[i]

print('One-hot:\n', one_hot)

One-hot:
 tensor([[0., 0., 1., 0.],
        [1., 0., 0., 0.],
        [0., 0., 0., 1.],
        [0., 1., 0., 0.]])


In [24]:
# Top-k scatter: put top-k values back into a full-size tensor
logits = torch.tensor([[0.1, 0.9, 0.3, 0.5, 0.7]])
k = 2

# Get top-k
vals, idx = torch.topk(logits, k, dim=-1)
print('Top-2 values:', vals)   # [0.9, 0.7]
print('Top-2 indices:', idx)   # [1, 4]

# Place back into -inf tensor
filtered = torch.full_like(logits, float('-inf'))
filtered.scatter_(dim=1, index=idx, src=vals)
print('Filtered:', filtered)  # [-inf, 0.9, -inf, -inf, 0.7]

Top-2 values: tensor([[0.9000, 0.7000]])
Top-2 indices: tensor([[1, 4]])
Filtered: tensor([[  -inf, 0.9000,   -inf,   -inf, 0.7000]])


**Exercise 5a**: Top-k filtering

In [25]:
logits = torch.randn(1, 20)
k = 5

# TODO: Get top-k values and indices
vals, idx = torch.topk(logits, k, dim=-1)

# TODO: Create -inf tensor, scatter top-k values back
filtered = torch.full_like(logits, float('-inf'))
# YOUR CODE: one line using scatter_
filtered.scatter_(dim=1, index=idx, src=vals)

assert (filtered != float('-inf')).sum() == k
print('✅ Top-k scatter works!')

✅ Top-k scatter works!
